In [8]:
%pip install pandas matplotlib seaborn pandapower grid

  Using cached argparse-1.4.0-py2.py3-none-any.whl.metadata (2.8 kB)
Using cached argparse-1.4.0-py2.py3-none-any.whl (23 kB)
Note: you may need to restart the kernel to use updated packages.


In [11]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

df = pd.read_csv("../simulation_results.csv")
print(f"Total records: {len(df)}")
print(f"Scenarios: {df['scenario'].unique()}")
print(f"Columns: {list(df.columns)}")
print(df.head(3))

Total records: 288
Scenarios: ['low_stress' 'medium_stress' 'high_stress']
Columns: ['timestep', 'scenario', 'ev_mw', 'dc_mw', 'total_mw', 'ev_bus_voltage_pu', 'ai_bus_voltage_pu', 'dist_bus_voltage_pu', 'transformer_loading_pct', 'n_violations_pre', 'voltage_violation', 'transformer_overload', 'optimization_triggered', 'optimization_feasible', 'opt_runtime_sec', 'demand_reduction_mw', 'optimized_total_mw', 'post_opt_ev_voltage', 'post_opt_ai_voltage', 'n_violations_post']
   timestep    scenario  ev_mw   dc_mw  total_mw  ev_bus_voltage_pu  \
0         0  low_stress  0.975  18.504    19.479             0.9774   
1         1  low_stress  6.024  18.875    24.899             0.9692   
2         2  low_stress  6.875  21.278    28.153             0.9645   

   ai_bus_voltage_pu  dist_bus_voltage_pu  transformer_loading_pct  \
0             0.9769               0.9775                     9.80   
1             0.9689               0.9695                    12.55   
2             0.9642       

In [12]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=False)
scenarios = ["low_stress", "medium_stress", "high_stress"]
colors = ["green", "orange", "red"]
titles = ["Low Stress", "Medium Stress", "High Stress"]

for idx, (scenario, color, title) in enumerate(zip(scenarios, colors, titles)):
    ax = axes[idx]
    data = df[df["scenario"] == scenario]
    ax.plot(data["timestep"], data["ev_bus_voltage_pu"],
            label="EV Bus", color=color, linewidth=2)
    ax.plot(data["timestep"], data["ai_bus_voltage_pu"],
            label="AI Bus", color=color, linewidth=2, linestyle="--")
    ax.axhline(y=0.95, color="red", linestyle=":", linewidth=1.5, label="V_min (0.95 pu)")
    ax.axhline(y=1.05, color="blue", linestyle=":", linewidth=1.5, label="V_max (1.05 pu)")
    ax.set_title(f"{title} — Bus Voltages Over Time", fontsize=12, fontweight="bold")
    ax.set_ylabel("Voltage (pu)")
    ax.set_ylim(0.0, 1.15)
    ax.legend(fontsize=9, loc="upper right")
    ax.grid(True, alpha=0.3)
    # Shade violation zones
    ax.fill_between(data["timestep"], 0.0, 0.95,
                    alpha=0.05, color="red", label="_nolegend_")

axes[-1].set_xlabel("Timestep (15-min intervals, 0=midnight)")
plt.suptitle("Distribution Grid Voltage Profiles Under High-Density Load Stress",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("../data/voltage_profiles.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: data/voltage_profiles.png")

Saved: data/voltage_profiles.png


/var/folders/s0/hpxkj67d4t78sywstk2c6sc00000gn/T/ipykernel_19234/1282786612.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [16]:
fig, ax = plt.subplots(figsize=(14, 5))

for scenario, color, title in zip(scenarios, colors, titles):
    data = df[df["scenario"] == scenario]
    ax.plot(data["timestep"], data["total_mw"],
            label=title, color=color, linewidth=2)
    ax.plot(data["timestep"], data["dc_mw"],
            color=color, linewidth=1, linestyle="--", alpha=0.6)

ax.set_title("Total Demand and Data Center Demand Over Time\n(solid = total, dashed = data center only)",
             fontsize=12, fontweight="bold")
ax.set_xlabel("Timestep (15-min intervals)")
ax.set_ylabel("Demand (MW)")
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("../data/demand_profiles.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: data/demand_profiles.png")

Saved: data/demand_profiles.png


/var/folders/s0/hpxkj67d4t78sywstk2c6sc00000gn/T/ipykernel_19234/1256814016.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [14]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for idx, (scenario, color, title) in enumerate(zip(scenarios, colors, titles)):
    ax = axes[idx]
    data = df[
        (df["scenario"] == scenario) &
        (df["optimization_triggered"] == True) &
        (df["post_opt_ev_voltage"].notna())
    ]
    if len(data) > 0:
        ax.scatter(data["timestep"], data["ev_bus_voltage_pu"],
                   color=color, alpha=0.6, s=40, label="Pre-optimization")
        ax.scatter(data["timestep"], data["post_opt_ev_voltage"],
                   color="blue", marker="^", s=40, alpha=0.6, label="Post-optimization")
        ax.axhline(y=0.95, color="red", linestyle="--",
                   linewidth=1.5, label="V_min (0.95)")
    ax.set_title(f"{title}", fontsize=11, fontweight="bold")
    ax.set_xlabel("Timestep")
    ax.set_ylabel("EV Bus Voltage (pu)")
    ax.set_ylim(0.0, 1.1)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle("EV Bus Voltage Before and After Optimization",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("../data/optimization_impact.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: data/optimization_impact.png")

Saved: data/optimization_impact.png


/var/folders/s0/hpxkj67d4t78sywstk2c6sc00000gn/T/ipykernel_19234/3963084880.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [15]:
summary = df.groupby("scenario").agg(
    avg_ev_voltage=("ev_bus_voltage_pu", "mean"),
    min_ev_voltage=("ev_bus_voltage_pu", "min"),
    max_ev_voltage=("ev_bus_voltage_pu", "max"),
    total_violations=("n_violations_pre", "sum"),
    optimizations_triggered=("optimization_triggered", "sum"),
    avg_demand_reduction_mw=("demand_reduction_mw", "mean"),
    avg_total_mw=("total_mw", "mean"),
    peak_total_mw=("total_mw", "max"),
).round(3)

# Reorder rows
order = ["low_stress", "medium_stress", "high_stress"]
summary = summary.reindex(order)

print("\n" + "="*70)
print("SIMULATION SUMMARY STATISTICS")
print("="*70)
print(summary.to_string())
print("="*70)

# Also print feasibility rates
print("\nOptimizer Feasibility Rates:")
for scenario in order:
    data = df[(df["scenario"] == scenario) & (df["optimization_triggered"] == True)]
    if len(data) > 0:
        feasible = data["optimization_feasible"].sum()
        rate = feasible / len(data) * 100
        print(f"  {scenario:<20} {feasible:.0f}/{len(data)} = {rate:.1f}%")


SIMULATION SUMMARY STATISTICS
               avg_ev_voltage  min_ev_voltage  max_ev_voltage  total_violations  optimizations_triggered  avg_demand_reduction_mw  avg_total_mw  peak_total_mw
scenario                                                                                                                                                      
low_stress              0.950           0.914           0.977               144                       50                   19.236        38.158         60.559
medium_stress           0.661           0.568           0.776               288                       96                  112.029       252.263        344.481
high_stress             0.295           0.204           0.465               288                       96                  337.152       983.059       1394.561

Optimizer Feasibility Rates:
  low_stress           50/50 = 100.0%
  medium_stress        96/96 = 100.0%
  high_stress          96/96 = 100.0%


In [17]:
fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=False)

for idx, (scenario, color, title) in enumerate(zip(scenarios, colors, titles)):
    ax = axes[idx]
    data = df[df["scenario"] == scenario]
    bars = ax.bar(data["timestep"], data["n_violations_pre"],
                  color=color, alpha=0.7, width=0.8)
    opt_times = data[data["optimization_triggered"] == True]["timestep"]
    ax.scatter(opt_times, [3.2] * len(opt_times),
               marker="v", color="black", s=20, zorder=5, label="Optimizer triggered")
    ax.set_title(f"{title} — Violations Per Timestep", fontsize=11, fontweight="bold")
    ax.set_ylabel("Violations")
    ax.set_ylim(0, 4)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3, axis="y")

axes[-1].set_xlabel("Timestep")
plt.suptitle("Grid Stability Violations and Optimization Triggers",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("../data/violations.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: data/violations.png")

Saved: data/violations.png


/var/folders/s0/hpxkj67d4t78sywstk2c6sc00000gn/T/ipykernel_19234/1390317534.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
